# Data Loading, Cleaning & Integrity Checks

Split out of `Untitled.ipynb`. This notebook is the entry point for the whole validation
pipeline: it loads every real-channel + Mock SDC collection, checks provenance, applies the
baseline-contamination fix, and re-checks the structural invariants that should hold by
construction (e.g. a row can't be both "mismatched" and "detected"). Save the cleaned
dataframes to disk at the end so the other notebooks (21-24) just load them, instead of
redoing this cleaning in every notebook.

**Not covered here** — reconciliation-fault modeling (point estimate vs. full distribution)
has its own dedicated notebook (`reconciliation_model_pointvsdist_cleaned.ipynb`), since it's
a large, separate piece of work with its own open questions.

In [1]:
import sys, ast, os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from qne.cascade.validation_utils import (
    is_baseline_contaminated, is_baseline_failure, mismatch_series, clean_fault_df,
)

RESULTS = PROJECT_DIR / "results"


## Load every collection

In [2]:
key_pairs_df = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))

files = {
    "toeplitz":              RESULTS / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv",
    "final_key":             RESULTS / "sdc_realchannel_finalkey_allkeys_asymptotic.csv",
    "reconciliation":        RESULTS / "sdc_realchannel_reconciliation_allkeys_asymptotic_fixed.csv",
    "verify_digest":         RESULTS / "sdc_realchannel_verifydigest_allkeys_final.csv",
    "toeplitz_fk":           RESULTS / "sdc_realchannel_toeplitz_allkeys_finitekey.csv",
    "final_key_fk":          RESULTS / "sdc_realchannel_finalkey_allkeys_finitekey.csv",
    "reconciliation_fk":     RESULTS / "sdc_realchannel_reconciliation_allkeys_finitekey.csv",
    "verify_digest_fk":      RESULTS / "sdc_realchannel_verifydigest_allkeys_finitekey.csv",
    "mock":                  RESULTS / "sdc_mock_allkeys_withverification.csv",
}

dfs = {}
for name, path in files.items():
    if not path.exists():
        print(f"!! {name}: FILE NOT FOUND at {path}")
        continue
    dfs[name] = pd.read_csv(str(path))
    print(f"{name}: {len(dfs[name])} rows, columns: {dfs[name].columns.tolist()}")


toeplitz: 400 rows, columns: ['toeplitz_prob', 'final_key_prob', 'reconciliation_prob', 'verify_digest_prob', 'seed', 'output_path', 'k_pe', 'total_corrections', 'secure_key_length', 't_verify', 'digest_length', 'verification_passed', 'remaining_errors_after_reconciliation', 'non_convergent', 'error', 'elapsed_seconds', 'faults_fired', 'length_mode', 'keys_match', 'alice_measurement_error', 'run', 'bob_output_path', 'alice_output_path', 'code_version', 'collection_timestamp', 'fault_type', 'prob', 'key_index', 'qber']
final_key: 400 rows, columns: ['toeplitz_prob', 'final_key_prob', 'reconciliation_prob', 'verify_digest_prob', 'seed', 'output_path', 'k_pe', 'total_corrections', 'secure_key_length', 't_verify', 'digest_length', 'verification_passed', 'remaining_errors_after_reconciliation', 'non_convergent', 'error', 'elapsed_seconds', 'faults_fired', 'length_mode', 'keys_match', 'alice_measurement_error', 'run', 'bob_output_path', 'alice_output_path', 'code_version', 'collection_timest

## Provenance sanity check

Before trusting any of this, confirm every collection actually has the fields you'd need to
know it's the *current* corrected version, not a stale run from before a fix.

In [3]:
for name, df in dfs.items():
    print(f"--- {name} ---")
    if "code_version" in df.columns:
        print(f"  code_version: {df['code_version'].unique()}")
    else:
        print("  NO code_version column -- can't confirm which fix-state this data is from")
    if "collection_timestamp" in df.columns:
        print(f"  collected: {df['collection_timestamp'].min()} to {df['collection_timestamp'].max()}")
    print(f"  key_index values: {sorted(df['key_index'].unique()) if 'key_index' in df.columns else 'MISSING'}")
    print(f"  length_mode: {df['length_mode'].unique() if 'length_mode' in df.columns else 'MISSING'}")


--- toeplitz ---
  code_version: <StringArray>
['unknown']
Length: 1, dtype: str
  collected: 2026-08-31T19:40:33.892058 to 2026-08-31T21:41:00.654444
  key_index values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  length_mode: <StringArray>
['asymptotic']
Length: 1, dtype: str
--- final_key ---
  code_version: <StringArray>
['unknown']
Length: 1, dtype: str
  collected: 2026-08-31T19:47:13.742270 to 2026-08-31T21:47:58.120693
  key_index values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  length_mode: <StringArray>
['asymptotic']
Length: 1, dtype: str
--- reconciliation ---
  NO code_version column -- can't confirm which fix-state this data is from
  key_index values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  length_mode: <StringArray>
['asymptotic']
Length: 1, dtype: str
--- verify_digest ---
  code_version: <StringArray>
['unknown']
Length: 1, dtype: str
  collected: 2026-08-31T20:01:23.358650 to 2026-08-31T22

## Baseline contamination

A run counts as baseline-contaminated if reconciliation still left residual errors even
though the failure you're attributing to the fault you're studying shouldn't include
ordinary Cascade residual-error leftovers. Apply and report per-key so you can see whether
it's concentrated on specific keys/probabilities or spread evenly.

In [4]:
clean_dfs = {}
for name in ["toeplitz", "final_key", "verify_digest", "toeplitz_fk", "final_key_fk", "verify_digest_fk"]:
    if name not in dfs:
        continue
    full, clean = clean_fault_df(dfs[name])
    dfs[name] = full
    clean_dfs[name] = clean
    n_contaminated = full["is_baseline_contaminated"].sum()
    print(f"{name}: {n_contaminated}/{len(full)} contaminated ({100*n_contaminated/len(full):.1f}%)")
    if n_contaminated > 0:
        print(full[full["is_baseline_contaminated"]].groupby("key_index").size().to_dict())

# reconciliation is NOT cleaned this way -- residual errors after a reconciliation-stage
# fault ARE the signal you're studying, not contamination
for name in ["reconciliation", "reconciliation_fk"]:
    if name in dfs:
        dfs[name]["mismatch"] = mismatch_series(dfs[name])
        clean_dfs[name] = dfs[name]

if "mock" in dfs:
    full, clean = clean_fault_df(dfs["mock"]) if "remaining_errors_after_reconciliation" in dfs["mock"].columns \
                  else (dfs["mock"].assign(mismatch=mismatch_series(dfs["mock"])), dfs["mock"].assign(mismatch=mismatch_series(dfs["mock"])))
    dfs["mock"] = full
    clean_dfs["mock"] = clean


toeplitz: 16/400 contaminated (4.0%)
{2: 8, 4: 8}
final_key: 16/400 contaminated (4.0%)
{2: 8, 4: 8}
verify_digest: 16/400 contaminated (4.0%)
{2: 8, 4: 8}
toeplitz_fk: 16/400 contaminated (4.0%)
{2: 8, 4: 8}
final_key_fk: 16/400 contaminated (4.0%)
{2: 8, 4: 8}
verify_digest_fk: 16/400 contaminated (4.0%)
{2: 8, 4: 8}


## Structural integrity checks

Two invariants that should hold by construction, and are worth re-checking on the *cleaned*
data specifically (a stale check on uncleaned data can hide a real violation behind
baseline-contamination noise, or manufacture a fake one).

In [5]:
print("=== Mutual exclusivity: a row can't be both mismatched AND detected ===")
for name in ["toeplitz", "final_key"]:
    if name not in clean_dfs:
        continue
    clean = clean_dfs[name]
    violations = clean[(clean["keys_match"] == False) & (clean["verification_passed"] == False)]
    print(f"{name}: {len(violations)} violating rows (should be 0)")
    if len(violations) > 0:
        print(violations[["key_index", "prob", "faults_fired", "keys_match", "verification_passed"]].head(10))

print("\n=== verify_digest: this fault type should never actually change the key ===")
if "verify_digest" in clean_dfs:
    df = clean_dfs["verify_digest"]
    n_mismatch = (df["keys_match"] == False).sum()
    print(f"{n_mismatch}/{len(df)} mismatches "
          f"({'OK -- structural result holds' if n_mismatch == 0 else 'VIOLATION -- investigate before presenting this fault type'})")
    if n_mismatch > 0:
        print(df[df['keys_match'] == False][["key_index", "prob", "faults_fired", "verification_passed"]])

print("\n=== reconciliation: keys_match should have real (non-null) values ===")
if "reconciliation" in dfs:
    vc = dfs["reconciliation"]["keys_match"].value_counts(dropna=False).to_dict()
    print(f"value counts: {vc} -- "
          f"{'OK' if dfs['reconciliation']['keys_match'].notna().any() else 'STILL BROKEN -- all None'}")


=== Mutual exclusivity: a row can't be both mismatched AND detected ===
toeplitz: 0 violating rows (should be 0)
final_key: 0 violating rows (should be 0)

=== verify_digest: this fault type should never actually change the key ===
0/384 mismatches (OK -- structural result holds)

=== reconciliation: keys_match should have real (non-null) values ===
value counts: {False: 246, True: 154} -- OK


## Runtime outliers (data-quality smell test, not a real analysis)

Long-tail run times sometimes indicate a hung/retried process rather than a real trial --
worth a glance before treating every row as equally trustworthy.

In [6]:
for name, df in dfs.items():
    if "elapsed_seconds" not in df.columns:
        continue
    median = df["elapsed_seconds"].median()
    outliers = df[df["elapsed_seconds"] > median * 2]
    print(f"{name}: median={median:.2f}s, {len(outliers)}/{len(df)} rows > 2x median")


toeplitz: median=4.06s, 0/400 rows > 2x median
final_key: median=3.93s, 0/400 rows > 2x median
reconciliation: median=3.87s, 0/400 rows > 2x median
verify_digest: median=3.75s, 0/400 rows > 2x median
toeplitz_fk: median=4.00s, 0/400 rows > 2x median
final_key_fk: median=3.98s, 0/400 rows > 2x median
reconciliation_fk: median=3.95s, 0/400 rows > 2x median
verify_digest_fk: median=3.74s, 0/400 rows > 2x median
mock: median=0.03s, 25/4800 rows > 2x median


## Save cleaned data for the other notebooks

In [7]:
for name, df in clean_dfs.items():
    out_path = RESULTS / f"{name}_clean.csv"
    df.to_csv(str(out_path), index=False)
    print(f"saved {out_path} ({len(df)} rows)")


saved /home/fabric/work/qkd-dependability/results/toeplitz_clean.csv (384 rows)
saved /home/fabric/work/qkd-dependability/results/final_key_clean.csv (384 rows)
saved /home/fabric/work/qkd-dependability/results/verify_digest_clean.csv (384 rows)
saved /home/fabric/work/qkd-dependability/results/toeplitz_fk_clean.csv (384 rows)
saved /home/fabric/work/qkd-dependability/results/final_key_fk_clean.csv (384 rows)
saved /home/fabric/work/qkd-dependability/results/verify_digest_fk_clean.csv (384 rows)
saved /home/fabric/work/qkd-dependability/results/reconciliation_clean.csv (400 rows)
saved /home/fabric/work/qkd-dependability/results/reconciliation_fk_clean.csv (400 rows)
saved /home/fabric/work/qkd-dependability/results/mock_clean.csv (4800 rows)
